In [1]:
import pandas as pd
import holidays

In [2]:
transactions = pd.read_csv("./data/amazon-purchases.csv")

In [3]:
transactions.head()

,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Category,Survey ResponseID
0,2018-12-04,7.98,1.0,NJ,SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...,B0143RTB1E,FLASH_MEMORY,R_01vNIayewjIIKMF
1,2018-12-22,13.99,1.0,NJ,Betron BS10 Earphones Wired Headphones in Ear ...,B01MA1MJ6H,HEADPHONES,R_01vNIayewjIIKMF
2,2018-12-24,8.99,1.0,NJ,NaN,B078JZTFN3,NaN,R_01vNIayewjIIKMF
3,2018-12-25,10.45,1.0,NJ,Perfecto Stainless Steel Shaving Bowl. Durable...,B06XWF9HML,DISHWARE_BOWL,R_01vNIayewjIIKMF
4,2018-12-25,10.00,1.0,NJ,Proraso Shaving Cream for Men,B00837ZOI0,SHAVING_AGENT,R_01vNIayewjIIKMF


In [4]:
print(transactions.head())

   Order Date  Purchase Price Per Unit  Quantity Shipping Address State  \
0  2018-12-04                     7.98       1.0                     NJ   
1  2018-12-22                    13.99       1.0                     NJ   
2  2018-12-24                     8.99       1.0                     NJ   
3  2018-12-25                    10.45       1.0                     NJ   
4  2018-12-25                    10.00       1.0                     NJ   

                                               Title ASIN/ISBN (Product Code)  \
0  SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...               B0143RTB1E   
1  Betron BS10 Earphones Wired Headphones in Ear ...               B01MA1MJ6H   
2                                                NaN               B078JZTFN3   
3  Perfecto Stainless Steel Shaving Bowl. Durable...               B06XWF9HML   
4                      Proraso Shaving Cream for Men               B00837ZOI0   

        Category  Survey ResponseID  
0   FLASH_MEMORY  R_01vN

In [5]:
transactions["Order Date"] = pd.to_datetime(transactions["Order Date"])

In [6]:
transactions_sorted = transactions.sort_values(["Survey ResponseID", "Order Date"], ascending=[True, True])

## Adding customer information

In [7]:
customers = pd.read_csv("./data/survey.csv")

### Formating Living state to match with Temperature Table
We use the living state of customer table to fill in missing values in the transaction table -> need to be in the same format

In [8]:
# Create a dictionary of state name mappings
state_abbrev = {
    'Alabama': 'AL',
    'Alaska': 'AK',
    'Arizona': 'AZ',
    'Arkansas': 'AR',
    'California': 'CA',
    'Colorado': 'CO',
    'Connecticut': 'CT',
    'Delaware': 'DE',
    'District of Columbia': 'DC',
    'Florida': 'FL',
    'Georgia': 'GA',
    'Hawaii': 'HI',
    'Idaho': 'ID',
    'Illinois': 'IL',
    'Indiana': 'IN',
    'Iowa': 'IA',
    'Kansas': 'KS',
    'Kentucky': 'KY',
    'Louisiana': 'LA',
    'Maine': 'ME',
    'Maryland': 'MD',
    'Massachusetts': 'MA',
    'Michigan': 'MI',
    'Minnesota': 'MN',
    'Mississippi': 'MS',
    'Missouri': 'MO',
    'Montana': 'MT',
    'Nebraska': 'NE',
    'Nevada': 'NV',
    'New Hampshire': 'NH',
    'New Jersey': 'NJ',
    'New Mexico': 'NM',
    'New York': 'NY',
    'North Carolina': 'NC',
    'North Dakota': 'ND',
    'Ohio': 'OH',
    'Oklahoma': 'OK',
    'Oregon': 'OR',
    'Pennsylvania': 'PA',
    'Rhode Island': 'RI',
    'South Carolina': 'SC',
    'South Dakota': 'SD',
    'Tennessee': 'TN',
    'Texas': 'TX',
    'Utah': 'UT',
    'Vermont': 'VT',
    'Virginia': 'VA',
    'Washington': 'WA',
    'West Virginia': 'WV',
    'Wisconsin': 'WI',
    'Wyoming': 'WY'
}

# Assuming your dataframe is called 'df' and the column with state names is called 'state'
customers["Q-demos-state"] = customers["Q-demos-state"].replace(state_abbrev)

### Mergin Dataframes

In [9]:
amazon_data = transactions.merge(customers, on="Survey ResponseID", how="left")

### Dealing with missing shipping addresses

Most of missing values in "Shipping Address State" can be filled by looking at "Q-demos-state" from the customer dataset -> we assume that the residence state is the same state as the shipping state 

In [10]:
amazon_data.loc[amazon_data["Shipping Address State"].isna(), "Shipping Address State"] = amazon_data["Q-demos-state"] # Replacing missing Shipping Addresses with living address from ordering customer

amazon_data.loc[amazon_data["Shipping Address State"] == "PR", "Shipping Address State"] = amazon_data["Q-demos-state"] # There is a shipping address state called PR which could be Puerto Rico, we replace this with the living state of the customer ordering

# Adding Temperature information

In [11]:
cols = ["date", "st_abb", "ppt", "tmax", "tmin", "tavg"]
weather = pd.read_csv("./Data/weather_data.csv", usecols=cols)
print(weather["st_abb"].nunique()) # No weather information for Alaska and Hawaii
weather.head()

48


,st_abb,date,ppt,tmax,tmin,tavg
0,AL,20170101,25.626,13.849,4.712,9.280
1,AL,20170102,40.827,17.970,12.968,15.469
2,AL,20170103,54.893,19.500,13.828,16.664
3,AL,20170104,0.048,19.158,8.744,13.951
4,AL,20170105,0.002,11.202,-0.147,5.527


In [12]:
weather["date"] = pd.to_datetime(weather["date"], format="%Y%m%d")
weather.rename(columns={"st_abb": "Shipping Address State"}, inplace=True)

## Creating trend free temperature on a state level

In [13]:
# Sort the dataframe by state and date
weather = weather.sort_values(['Shipping Address State', 'date'])

# Calculate the temperature difference for each state vs the previous year
weather['Temp_No_Season'] = weather.groupby('Shipping Address State')['tavg'].diff(365).round(3)

In [14]:
# Joining weather

data_combined = pd.merge(
    left=weather, 
    right=amazon_data,
    how='left',
    left_on=['Shipping Address State', 'date'],
    right_on=['Shipping Address State', 'Order Date'],
)

In [15]:
# Check for which states we have no temperature information
# AK, DC, HI, I did not reside in the United States
# We still leave them in the dataset for completnes 

data_grouped = data_combined.groupby("Shipping Address State")

### Calculate Revenue

In [16]:
data_combined["Revenue"] = data_combined["Purchase Price Per Unit"] * data_combined["Quantity"]

### Remove the one value for 2024

In [17]:
data_combined  = data_combined[(data_combined['date'] < '2024-01-01')]

## CHECKPOINT: Saving complete combined dataset

In [18]:
data_combined.to_csv('./data/data_combined_full.csv')

# Process data for next steps 
Change date to dtype
Renaming date column to Order Date

In [19]:
# Needed Columns for the Hypothesis testing
data_needed = ["date", "Category", "Revenue", "Shipping Address State", "tavg", "ppt", "Survey ResponseID"]

transactions_temp = pd.read_csv("./Data/data_combined_full.csv", usecols=data_needed)
transactions_temp["date"] = pd.to_datetime(transactions_temp["date"]) # Formating Order date back to Datetime dtype, in read_csv you cant set column to datetime so need to do manually

transactions_temp.rename(columns={"date": "Order Date"}, inplace=True)

### Creating an Dataframe with all dates and States

In [20]:
states = transactions_temp['Shipping Address State'].unique()

date_range = pd.date_range(
    start=transactions_temp['Order Date'].min(),
    end=transactions_temp['Order Date'].max(),
    freq='D'
)
print(transactions_temp['Order Date'].min())

date_state_combinations = pd.MultiIndex.from_product(
    [date_range, states],
    names=['Order Date', 'Shipping Address State']
)
full_index_df = pd.DataFrame(index=date_state_combinations).reset_index()



2017-01-01 00:00:00


### Definition of Categories

In [21]:
Fashion = [
    "ACCESSORY", "ADULT_COSTUME", "APPAREL", "APPAREL_BELT", "APPAREL_GLOVES",
    "APPAREL_HEAD_NECK_COVERING", "APPAREL_PIN", "APPENDAGE_WARMER", "APRON",
    "ARM_SLEEVE", "Apparel", "BABY_JUMPER_WALKER", "BELTS", "BLAZER",
    "BLUE_LIGHT_BLOCKING_EYEGLASSES", "BODYSTOCKING", "BOOT", "BRA", "BRACELET",
    "BRA_UNDERWEAR_SET", "CHOLI", "COAT", "COORDINATED_OUTFIT", "CORRECTIVE_EYEGLASSES",
    "CORSET", "DRESS", "DUPATTA", "EARMUFF", "EARRING", "ETHNIC_WEAR",
    "FASHIONEARRING", "FASHIONNECKLACEBRACELETANKLET", "FASHIONOTHER", "FASHIONRING",
    "FASHION_JEWELRY", "FINEEARRING", "FINENECKLACEBRACELETANKLET", "FINERING",
    "FOOTWEAR", "GUILD_APPAREL", "GUILD_JEWELRY", "GUILD_SHOES", "HAT", "HOSIERY",
    "JEWELRY", "JEWELRY_SET", "KIMONO", "KURTA", "LEG_SLEEVE", "LEOTARD",
    "MOISTURIZING_SOCK_GLOVE", "NECKLACE", "NECKTIE", "NIGHTGOWN_NIGHTSHIRT",
    "ORCA_SHIRT", "OUTERWEAR", "OVERALLS", "PAJAMAS", "PANTS", "PIERCING_JEWELRY",
    "ROBE", "SANDAL", "SAREE", "SHIRT", "SHOES", "SHORTS", "SKIRT", "SLEEPWEAR",
    "SLIPPER", "SNOWSHOE", "SNOWSUIT", "SNOW_PANT", "SOCK", "SOCKSHOSIERY", "SUIT",
    "SUNGLASSES", "SUSPENDER", "SWEATER", "SWEATSHIRT", "SWIMWEAR", "SWIM_CAP",
    "TIGHTS", "TRACK_SUIT", "UNDERPANTS", "UNDERWEAR", "UNDERGARMENT_SLIP",
    "UNDERGARMENT_THIGH_SLIMMER", "UNION_SUIT", "VEST", "WATCH", "WATCHES", "WATCH_BAND"
]


Groceries = [
   "ALCOHOLIC_BEVERAGE", "BABY_FOOD", "BABY_FORMULA", "BAKING_CHOCOLATE", "BEER",
   "BEVERAGE", "BREAD", "BREAKFAST_CEREAL", "CAKE", "CANDY", "CEREAL",
   "CHOCOLATE_CANDY", "COFFEE", "CONDIMENT", "COOKIE", "CRACKER", "CULINARY_SALT",
   "DAIRY_BASED_BUTTER", "DAIRY_BASED_CHEESE", "DAIRY_BASED_CREAM",
   "DAIRY_BASED_DRINK", "DAIRY_BASED_ICE_CREAM", "DAIRY_BASED_PUDDING",
   "DAIRY_BASED_YOGURT", "DONUT", "DRINK_FLAVORED", "EDIBLE_OIL_VEGETABLE",
   "FLOUR", "FRUIT", "FRUIT_SNACK", "FUDGE", "GOURMET_FOOD", "GROCERY", "Grocery",
   "HEALTH_FOOD", "HONEY", "JERKY", "JUICE_AND_JUICE_DRINK", "LEGUME",
   "MEAL_REPLACEMENT_BEVERAGE", "MEAT_ALTERNATIVE", "MILK_SUBSTITUTE",
   "NON_DAIRY_CHEESE", "NON_DAIRY_CREAM", "NON_DAIRY_ICE_CREAM",
   "NON_DAIRY_PUDDING", "NON_DAIRY_YOGURT", "NOODLE", "NUTS", "NUT_AND_SEED",
   "NUT_BUTTER", "OLIVE", "PACKAGED_SOUP_AND_STEW", "PASTRY", "POPCORN",
   "PRETZEL", "PROTEIN_DRINK", "PROTEIN_SUPPLEMENT_POWDER", "PUFFED_SNACK",
   "RICE_MIX", "RICE_WINE", "SALAD_DRESSING", "SEAFOOD", "SEASONING",
   "SNACK_CHIP_AND_CRISP", "SNACK_FOOD", "SNACK_FOOD_BAR", "SNACK_MIX",
   "SPORTS_DRINK", "SUGAR", "SUGAR_CANDY", "SUGAR_SUBSTITUTE", "SYRUP", "TEA",
   "TOFU", "VEGETABLE", "VEGETARIAN_EGG_SUBSTITUTE", "WATER"
]

In [22]:
mask_fashion = transactions_temp["Category"].isin(Fashion)
Fashion_transactions = transactions_temp[mask_fashion]
Fashion_transactions = Fashion_transactions.sort_values(["Order Date"]).copy() # .copy() to create new dataframe in memory and not a pd view


mask_groceries = transactions_temp["Category"].isin(Groceries)
Grocerie_transactions = transactions_temp[mask_groceries]
Grocerie_transactions = Grocerie_transactions.sort_values(["Order Date"]).copy()

### Definition of Sub-Categories

In [23]:
def add_weather_category_fashion(df, category_column): 
    category_mapping = {
    # Rain/Snow Protection
    'rain_protection': [
        'FOOTWEAR', 'SHOES', 'GUILD_SHOES'
    ],
    
    # Seasonal (Cold)
    'cold_weather': [
        'SWEATER', 'SWEATSHIRT', 'APPENDAGE_WARMER',
        'ARM_SLEEVE', 'LEG_SLEEVE', 'EARMUFF',
        'APPAREL_GLOVES', 'MOISTURIZING_SOCK_GLOVE',
        'APPAREL_HEAD_NECK_COVERING', 'SOCK', 'SOCKSHOSIERY',
        'HOSIERY', 'TIGHTS', 'BOOT', 'COAT', 'OUTERWEAR', 'SNOW_PANT', 'SNOWSUIT', 'SNOWSHOE'
    ],
    
    # Seasonal (Hot)
    'hot_weather': [
        'SWIMWEAR', 'SWIM_CAP', 'SHORTS', 'SANDAL', 'SUNGLASSES', 'ROBE'
    ],
    
    # Other: Cultural/Ethnic Wear
    'other': [
        'ETHNIC_WEAR', 'KURTA', 'SAREE', 'DUPATTA',
        'CHOLI'
    ],

    # Indoor/Basic Wear, Sleepwear, Undergarments, Layering -> not weather dependent
    'indoor_basic': [
        'SHIRT', 'ORCA_SHIRT', 'PANTS', 'DRESS', 'SKIRT', 
        'LEOTARD', 'OVERALLS', 'APRON', 'GUILD_APPAREL',
        'APPAREL', 'Apparel', 'NECKTIE', 'CORSET',
        'BABY_JUMPER_WALKER', 'PAJAMAS', 'NIGHTGOWN_NIGHTSHIRT',
        'SLEEPWEAR', 'SLIPPER', 'BODYSTOCKING', 'BRA', 'BRA_UNDERWEAR_SET', 'UNDERWEAR', 'UNDERPANTS',
        'UNDERGARMENT_SLIP', 'UNDERGARMENT_THIGH_SLIMMER',
        'VEST', 'BLAZER', 'TRACK_SUIT', 'UNION_SUIT',
        'SUIT', 'COORDINATED_OUTFIT', 'KIMONO'
    ],
    
    # Accessories (Fashion), Jewelry, Eyewear & Watches
    'fashion_accessories': [
        'ACCESSORY', 'APPAREL_BELT', 'APPAREL_PIN', 'BELTS',
        'SUSPENDER', 'HAT', 'ADULT_COSTUME', 'JEWELRY', 'JEWELRY_SET', 'BRACELET', 'EARRING', 
        'NECKLACE', 'PIERCING_JEWELRY', 'FASHIONEARRING',
        'FASHIONNECKLACEBRACELETANKLET', 'FASHIONOTHER', 
        'FASHIONRING', 'FASHION_JEWELRY', 'FINEEARRING',
        'FINENECKLACEBRACELETANKLET', 'FINERING', 'GUILD_JEWELRY', 'WATCH', 'WATCHES', 'WATCH_BAND',
        'CORRECTIVE_EYEGLASSES', 'BLUE_LIGHT_BLOCKING_EYEGLASSES'
    ]
}
    
    # Create the reverse mapping
    category_to_weather = {}
    for weather_cat, categories in category_mapping.items():
        for category in categories:
            category_to_weather[category] = weather_cat
    
    # Add the new column
    Fashion_transactions['weather_category'] = Fashion_transactions[category_column].map(category_to_weather)
    
    
    return df

# Example usage:
Fashion_transactions = add_weather_category_fashion(Fashion_transactions, 'Category')

In [24]:
def add_weather_category_grocery(df, category_column):
    category_mapping = {
        # Hot Weather Sensitive (items with higher demand in hot weather)
        'hot_weather': [
            'WATER', 'SPORTS_DRINK', 'BEVERAGE', 'DRINK_FLAVORED',
            'JUICE_AND_JUICE_DRINK', 'DAIRY_BASED_ICE_CREAM',
            'NON_DAIRY_ICE_CREAM', 'PROTEIN_DRINK',
            'MEAL_REPLACEMENT_BEVERAGE'
        ],
        
        # Cold Weather Sensitive (items with higher demand in cold weather)
        'cold_weather': [
            'PACKAGED_SOUP_AND_STEW', 'COFFEE', 'TEA',
            'HOT_CHOCOLATE', 'RICE_MIX'
        ],
        
        # Fresh/Perishable (items affected by temperature/humidity)
        'fresh_perishable': [
            'DAIRY_BASED_BUTTER', 'DAIRY_BASED_CHEESE', 'DAIRY_BASED_CREAM',
            'DAIRY_BASED_YOGURT', 'NON_DAIRY_CHEESE', 'NON_DAIRY_CREAM',
            'NON_DAIRY_YOGURT', 'FRUIT', 'VEGETABLE', 'SEAFOOD',
            'TOFU', 'DAIRY_BASED_PUDDING', 'NON_DAIRY_PUDDING', "DAIRY_BASED_DRINK"
        ],
        
        # Shelf-Stable Pantry (minimal weather impact)
        'shelf_stable_pantry': [
            'FLOUR', 'SUGAR', 'SUGAR_SUBSTITUTE', 'HONEY', 'SYRUP',
            'CULINARY_SALT', 'EDIBLE_OIL_VEGETABLE', 'SEASONING',
            'CONDIMENT', 'SALAD_DRESSING', 'BAKING_CHOCOLATE',
            'PROTEIN_SUPPLEMENT_POWDER', 'NOODLE', 'OLIVE'
        ],
        
        # Snacks & Confectionery
        'snacks_confectionery': [
            'CANDY', 'CHOCOLATE_CANDY', 'SUGAR_CANDY', 'FUDGE',
            'SNACK_CHIP_AND_CRISP', 'POPCORN', 'PRETZEL',
            'PUFFED_SNACK', 'SNACK_MIX', 'SNACK_FOOD',
            'FRUIT_SNACK', 'COOKIE', 'CRACKER', 'JERKY',
            'NUTS', 'NUT_AND_SEED', 'NUT_BUTTER',
            'SNACK_FOOD_BAR'
        ],
        
        # Baked Goods (humidity sensitive)
        'baked_goods': [
            'BREAD', 'CAKE', 'PASTRY', 'DONUT'
        ],
        
        # Breakfast & Cereal
        'breakfast_cereal': [
            'BREAKFAST_CEREAL', 'CEREAL'
        ],
        
        # Baby Products
        'baby_products': [
            'BABY_FOOD', 'BABY_FORMULA'
        ],
        
        # Specialty & Health
        'specialty_health': [
            'GOURMET_FOOD', 'HEALTH_FOOD', 'MEAT_ALTERNATIVE',
            'MILK_SUBSTITUTE', 'VEGETARIAN_EGG_SUBSTITUTE',
            'LEGUME'
        ],
        
        # Alcoholic Beverages
        'alcoholic_beverages': [
            'ALCOHOLIC_BEVERAGE', 'BEER', 'RICE_WINE'
        ]
    }
    
    # Create the reverse mapping
    category_to_weather = {}
    for weather_cat, categories in category_mapping.items():
        for category in categories:
            category_to_weather[category] = weather_cat
            
    # Handle generic grocery categories
    category_to_weather['GROCERY'] = 'general_grocery'
    category_to_weather['Grocery'] = 'general_grocery'
    
    # Add the new column
    df['weather_category'] = df[category_column].map(category_to_weather)
    
    
    return df

# Example usage:
Grocerie_transactions = add_weather_category_grocery(Grocerie_transactions, 'Category')

In [25]:
Grocerie_transactions.to_csv('./data/groceries_meta_level.csv')
Fashion_transactions.to_csv('./data/fashion_meta_level.csv')

In [26]:
Grocerie_transactions = Grocerie_transactions.sort_values("Order Date", ascending=True)

Grocerie_transactions_grouped = Grocerie_transactions.groupby(["Order Date", "Shipping Address State"]).agg({"Revenue": "sum", "Survey ResponseID": lambda x: list(set(x)), "weather_category": lambda x: list(set(x))})
Grocerie_transactions_grouped["Revenue"] = Grocerie_transactions_grouped["Revenue"].round(2)
Grocerie_transactions_grouped.reset_index(["Order Date", "Shipping Address State"], inplace=True)

In [27]:
demographic_df = pd.read_csv("./data/survey.csv")


Fashion_transactions_merged = pd.merge(
    left=Fashion_transactions, 
    right=demographic_df,
    how='left',
    left_on=['Survey ResponseID'],
    right_on=['Survey ResponseID'],
)


Grocerie_transactions_merged = pd.merge(
    left=Grocerie_transactions, 
    right=demographic_df,
    how='left',
    left_on=['Survey ResponseID'],
    right_on=['Survey ResponseID'],
)


Grocerie_transactions_merged.head()

,Shipping Address State,Order Date,ppt,tavg,Category,Survey ResponseID,Revenue,weather_category,Q-demos-age,Q-demos-hispanic,...,Q-substance-use-marijuana,Q-substance-use-alcohol,Q-personal-diabetes,Q-personal-wheelchair,Q-life-changes,Q-sell-YOUR-data,Q-sell-consumer-data,Q-small-biz-use,Q-census-use,Q-research-society
0,NM,2018-01-01,0.000,-0.210,NON_DAIRY_CREAM,R_Wflh7jXpugM20Dv,11.30,fresh_perishable,65 and older,No,...,Yes,No,No,No,NaN,No,No,No,Yes,Yes
1,IN,2018-01-01,0.187,-15.906,NUT_AND_SEED,R_5zP15CubGsMclkR,7.98,snacks_confectionery,25 - 34 years,No,...,Yes,No,No,No,NaN,Yes if I get part of the profit,Yes if consumers get part of the profit,Yes,Yes,Yes
2,CA,2018-01-01,0.003,10.109,COFFEE,R_1LYvldXrvEt6Cel,33.00,cold_weather,65 and older,No,...,No,No,No,No,NaN,No,No,No,No,No
3,FL,2018-01-01,2.699,12.016,LEGUME,R_3rHSccsh0mlkeuy,11.99,specialty_health,35 - 44 years,No,...,No,No,No,No,NaN,I don't know,I don't know,I don't know,I don't know,Yes
4,FL,2018-01-01,2.699,12.016,DAIRY_BASED_DRINK,R_3rHSccsh0mlkeuy,9.95,fresh_perishable,35 - 44 years,No,...,No,No,No,No,NaN,I don't know,I don't know,I don't know,I don't know,Yes


### Processing to only have one entry per state and date

1. Creating Subcategories Shares of Revenue
2. Modus for Categorical variables 

In [28]:
# First get ratios by category
category_ratios = Fashion_transactions_merged.groupby(
    ["Order Date", "Shipping Address State", "weather_category"]
)['Revenue'].sum().reset_index()

# Calculate ratios within each date-state group
category_ratios['ratio'] = category_ratios.groupby(
    ["Order Date", "Shipping Address State"]
)['Revenue'].transform(lambda x: (x / x.sum()).round(4))

# Pivot to get one column per category ratio
ratio_columns = category_ratios.pivot_table(
    index=["Order Date", "Shipping Address State"],
    columns="weather_category",
    values="ratio",
    fill_value=0
).reset_index()

# Rename ratio columns
ratio_columns.columns = ['Order Date', 'Shipping Address State'] + [
    f'ratio_{col}' for col in ratio_columns.columns[2:]
]

# Get other aggregated info (revenue, survey IDs, categories)
other_info = Fashion_transactions_merged.groupby(["Order Date", "Shipping Address State"]).agg({
    "Revenue": "sum",
    "Survey ResponseID": lambda x: list(set(x)),
    "weather_category": lambda x: list(set(x)),
    "Q-demos-age": lambda x: x.mode()[0] if not x.mode().empty else None, # modus on categorical columns
    "Q-demos-race": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-demos-education": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-demos-income": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-demos-gender": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-amazon-use-how-oft": lambda x: x.mode()[0] if not x.mode().empty else None
}).reset_index()

other_info = other_info.rename(columns={
    "Q-demos-age": "age",
    "Q-demos-race": "race",
    "Q-demos-education": "education",
    "Q-demos-income": "income",
    "Q-demos-gender": "gender",
    "Q-amazon-use-how-oft": "amazon_frequency"
})


# Merge ratio columns with other info
Fashion_transactions_grouped = pd.merge(
    other_info,
    ratio_columns,
    on=["Order Date", "Shipping Address State"]
)

# Round revenue
Fashion_transactions_grouped["Revenue"] = Fashion_transactions_grouped["Revenue"].round(2)

In [29]:
# First get ratios by category
category_ratios = Grocerie_transactions_merged.groupby(
    ["Order Date", "Shipping Address State", "weather_category"]
)['Revenue'].sum().reset_index()

# Calculate ratios within each date-state group
category_ratios['ratio'] = category_ratios.groupby(
    ["Order Date", "Shipping Address State"]
)['Revenue'].transform(lambda x: (x / x.sum()).round(4))

# Pivot to get one column per category ratio
ratio_columns = category_ratios.pivot_table(
    index=["Order Date", "Shipping Address State"],
    columns="weather_category",
    values="ratio",
    fill_value=0
).reset_index()

# Rename ratio columns
ratio_columns.columns = ['Order Date', 'Shipping Address State'] + [
    f'ratio_{col}' for col in ratio_columns.columns[2:]
]

# Get other aggregated info (revenue, survey IDs, categories)
other_info = Grocerie_transactions_merged.groupby(["Order Date", "Shipping Address State"]).agg({
    "Revenue": "sum",
    "Survey ResponseID": lambda x: list(set(x)),
    "weather_category": lambda x: list(set(x)),
    "Q-demos-age": lambda x: x.mode()[0] if not x.mode().empty else None, # modus on categorical columns
    "Q-demos-race": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-demos-education": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-demos-income": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-demos-gender": lambda x: x.mode()[0] if not x.mode().empty else None,
    "Q-amazon-use-how-oft": lambda x: x.mode()[0] if not x.mode().empty else None
}).reset_index()

other_info = other_info.rename(columns={
    "Q-demos-age": "age",
    "Q-demos-race": "race",
    "Q-demos-education": "education",
    "Q-demos-income": "income",
    "Q-demos-gender": "gender",
    "Q-amazon-use-how-oft": "amazon_frequency"
})


# Merge ratio columns with other info
Grocerie_transactions_merged = pd.merge(
    other_info,
    ratio_columns,
    on=["Order Date", "Shipping Address State"]
)

# Round revenue
Grocerie_transactions_merged["Revenue"] = Grocerie_transactions_merged["Revenue"].round(2)

##### Adding weekday information

In [30]:
Fashion_transactions_grouped["day_of_the_week"] = Fashion_transactions_grouped['Order Date'].dt.day_name()

# get statelevel holidays
def get_holiday(date, state):
    us_holidays = holidays.US(state=state, years=date.year)
    return True if date in us_holidays else None  # Returns true if date is holiday or None if not a holiday

# get season
def get_season(date):
    month = date.month
    day = date.day
    
    # Spring: March 20 - June 20
    if (month == 3 and day >= 20) or month == 4 or month == 5 or (month == 6 and day <= 20):
        return "Spring"
    # Summer: June 21 - September 22
    elif (month == 6 and day >= 21) or month == 7 or month == 8 or (month == 9 and day <= 22):
        return "Summer"
    # Fall: September 23 - December 20
    elif (month == 9 and day >= 23) or month == 10 or month == 11 or (month == 12 and day <= 20):
        return "Fall"
    # Winter: December 21 - March 19
    else:
        return "Winter"


Fashion_transactions_grouped['Holiday'] = Fashion_transactions_grouped.apply(
    lambda row: get_holiday(row['Order Date'], row['Shipping Address State']), axis=1
)

Fashion_transactions_grouped['Season'] = Fashion_transactions_grouped['Order Date'].apply(get_season)

In [31]:
Grocerie_transactions_merged["day_of_the_week"] = Grocerie_transactions_merged['Order Date'].dt.day_name()

# get statelevel holidays
def get_holiday(date, state):
    us_holidays = holidays.US(state=state, years=date.year)
    return True if date in us_holidays else None  # Returns true if date is holiday or None if not a holiday

# get season
def get_season(date):
    month = date.month
    day = date.day
    
    # Spring: March 20 - June 20
    if (month == 3 and day >= 20) or month == 4 or month == 5 or (month == 6 and day <= 20):
        return "Spring"
    # Summer: June 21 - September 22
    elif (month == 6 and day >= 21) or month == 7 or month == 8 or (month == 9 and day <= 22):
        return "Summer"
    # Fall: September 23 - December 20
    elif (month == 9 and day >= 23) or month == 10 or month == 11 or (month == 12 and day <= 20):
        return "Fall"
    # Winter: December 21 - March 19
    else:
        return "Winter"


Grocerie_transactions_merged['Holiday'] = Grocerie_transactions_merged.apply(
    lambda row: get_holiday(row['Order Date'], row['Shipping Address State']), axis=1
)

Grocerie_transactions_merged['Season'] = Grocerie_transactions_merged['Order Date'].apply(get_season)

## Joining dataframes

## Groceries

In [32]:
groceries_combined = pd.merge(
    left=full_index_df, 
    right=Grocerie_transactions_merged,
    how='left',
    left_on=['Shipping Address State', 'Order Date'],
    right_on=['Shipping Address State', 'Order Date'],
)

groceries_combined_final = pd.merge(
    left= groceries_combined, 
    right= weather,
    how='left',
    left_on=['Shipping Address State', 'Order Date'],
    right_on=['Shipping Address State', 'date'],
)

groceries_combined_final = groceries_combined_final.drop(columns=["date"])

groceries_combined_final["Revenue"] = groceries_combined_final['Revenue'].fillna(0)

groceries_combined_final["Survey ResponseID"] = groceries_combined_final["Survey ResponseID"].fillna("")

groceries_combined_final  = groceries_combined_final[(groceries_combined_final['Order Date'] < '2023-03-21')]

In [33]:
groceries_combined_final

,Order Date,Shipping Address State,Revenue,Survey ResponseID,weather_category,age,race,education,income,gender,...,ratio_snacks_confectionery,ratio_specialty_health,day_of_the_week,Holiday,Season,ppt,tmax,tmin,tavg,Temp_No_Season
0,2017-01-01,AL,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,25.626,13.849,4.712,9.280,NaN
1,2017-01-01,AR,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.999,10.667,4.430,7.548,NaN
2,2017-01-01,AZ,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.824,9.969,4.154,7.061,NaN
3,2017-01-01,CA,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.760,9.559,0.137,4.848,NaN
4,2017-01-01,CO,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.385,2.787,-9.496,-3.354,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108955,2023-03-20,VT,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.110,-3.244,-9.830,-6.537,-13.007
108956,2023-03-20,WA,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.920,11.746,0.634,6.190,2.887
108957,2023-03-20,WI,0.00,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.000,2.895,-11.230,-4.168,-6.441
108958,2023-03-20,WV,73.69,[R_8CXcySQawXPuJwJ],"[cold_weather, breakfast_cereal, snacks_confec...",35 - 44 years,White or Caucasian,High school diploma or GED,"$25,000 - $49,999",Female,...,0.393,0.0,Monday,None,Spring,0.000,0.404,-9.255,-4.426,-15.861


In [34]:
groceries_combined_final.to_csv('./data/grocery_data.csv')

## Fashion

In [35]:
fashion_combined = pd.merge(
    left=full_index_df, 
    right=Fashion_transactions_grouped,
    how='left',
    left_on=['Shipping Address State', 'Order Date'],
    right_on=['Shipping Address State', 'Order Date'],
)


fashion_combined_final = pd.merge(
    left= fashion_combined, 
    right= weather,
    how='left',
    left_on=['Shipping Address State', 'Order Date'],
    right_on=['Shipping Address State', 'date'],
)

fashion_combined_final = fashion_combined_final.drop(columns=["date"])

fashion_combined_final["Revenue"] = fashion_combined_final['Revenue'].fillna(0)

fashion_combined_final["Survey ResponseID"] = fashion_combined_final["Survey ResponseID"].fillna("")

fashion_combined_final  = fashion_combined_final[(fashion_combined_final['Order Date'] < '2023-03-21')]

In [36]:
fashion_combined_final.head()

,Order Date,Shipping Address State,Revenue,Survey ResponseID,weather_category,age,race,education,income,gender,...,ratio_other,ratio_rain_protection,day_of_the_week,Holiday,Season,ppt,tmax,tmin,tavg,Temp_No_Season
0,2017-01-01,AL,0.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,25.626,13.849,4.712,9.280,NaN
1,2017-01-01,AR,0.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.999,10.667,4.430,7.548,NaN
2,2017-01-01,AZ,0.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,4.824,9.969,4.154,7.061,NaN
3,2017-01-01,CA,0.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.760,9.559,0.137,4.848,NaN
4,2017-01-01,CO,0.0,,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.385,2.787,-9.496,-3.354,NaN


In [37]:
fashion_combined_final.to_csv('./data/fashion_data.csv')

# Summary Statistics

In [38]:
print("Fashion df: \n")
print(fashion_combined_final[["Revenue", "ratio_cold_weather", "ratio_hot_weather", "ppt", "tavg", "Temp_No_Season"]].describe().round(3))

Fashion df: 

          Revenue  ratio_cold_weather  ratio_hot_weather         ppt  \
count  108960.000           46888.000          46888.000  108960.000   
mean       35.910               0.138              0.111       2.865   
std        74.074               0.280              0.253       5.924   
min         0.000               0.000              0.000       0.000   
25%         0.000               0.000              0.000       0.010   
50%         0.000               0.000              0.000       0.375   
75%        39.990               0.118              0.000       2.924   
max      2013.460               1.000              1.000     137.215   

             tavg  Temp_No_Season  
count  108960.000       91440.000  
mean       11.403          -0.029  
std        10.765           5.961  
min       -32.310         -29.664  
25%         3.312          -3.465  
50%        12.136          -0.017  
75%        20.569           3.417  
max        33.215          32.108  


In [39]:
print("Groceries df: \n")
print(groceries_combined_final[["Order Date", "Revenue", "ratio_cold_weather", "ratio_hot_weather", "ppt", "tavg", "Temp_No_Season"]].describe().round(3))

Groceries df: 

                          Order Date     Revenue  ratio_cold_weather  \
count                         108960  108960.000           38753.000   
mean   2020-02-09 11:59:59.999999744      19.381               0.189   
min              2017-01-01 00:00:00       0.000               0.000   
25%              2018-07-22 00:00:00       0.000               0.000   
50%              2020-02-09 12:00:00       0.000               0.000   
75%              2021-08-30 00:00:00      20.990               0.259   
max              2023-03-20 00:00:00    1916.800               1.000   
std                              NaN      43.060               0.324   

       ratio_hot_weather         ppt        tavg  Temp_No_Season  
count          38753.000  108960.000  108960.000       91440.000  
mean               0.164       2.865      11.403          -0.029  
min                0.000       0.000     -32.310         -29.664  
25%                0.000       0.010       3.312          -3.465  
